# 01 — Exploratory Data Analysis: Elliptic Bitcoin Dataset

**ETGT-FRD Project** | REVA University Mini Project

This notebook provides a comprehensive exploratory analysis of the three raw CSV files:
- `elliptic_txs_features.csv` — 165 node features per transaction
- `elliptic_txs_classes.csv` — ground-truth labels (illicit/licit/unknown)
- `elliptic_txs_edgelist.csv` — directed payment edges

In [ ]:
import sys
from pathlib import Path

# Add project root to path
ROOT = Path().absolute().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import networkx as nx
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='husl')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

DATA_DIR = ROOT / 'data' / 'raw'
FIG_DIR  = ROOT / 'outputs' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {ROOT}')
print(f'Data directory: {DATA_DIR}')

## 1. Load Raw CSV files

In [ ]:
# --- Features ---
feat_df = pd.read_csv(DATA_DIR / 'elliptic_txs_features.csv', header=None)
feat_df.columns = ['txId', 'time_step'] + [f'feat_{i}' for i in range(feat_df.shape[1] - 2)]

# --- Classes ---
class_df = pd.read_csv(DATA_DIR / 'elliptic_txs_classes.csv')
class_df.columns = ['txId', 'class']

# --- Edges ---
edge_df = pd.read_csv(DATA_DIR / 'elliptic_txs_edgelist.csv')
edge_df.columns = ['txId1', 'txId2']

print(f'Features shape : {feat_df.shape}')
print(f'Classes shape  : {class_df.shape}')
print(f'Edges shape    : {edge_df.shape}')
feat_df.head(3)

## 2. Label Distribution

In [ ]:
label_counts = class_df['class'].value_counts()
print('Label distribution:')
print(label_counts)
print()

# Map labels
label_map = {'1': 'Illicit', '2': 'Licit', 'unknown': 'Unknown'}
label_counts.index = [label_map.get(str(i), str(i)) for i in label_counts.index]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#E84855', '#2DCC70', '#95A5A6']

# Bar chart
axes[0].bar(label_counts.index, label_counts.values, color=colors, edgecolor='white', width=0.5)
axes[0].set_title('Transaction Label Counts', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

# Pie chart (labelled only)
labelled = label_counts[['Illicit', 'Licit']]
axes[1].pie(labelled.values, labels=labelled.index, colors=['#E84855', '#2DCC70'],
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 13})
axes[1].set_title('Labelled Class Ratio (Illicit vs Licit)', fontweight='bold')

plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_label_distribution.png', bbox_inches='tight')
plt.show()
print('✅ Class imbalance clearly visible — motivates Focal Loss')

## 3. Temporal Analysis — Transactions per Time Step

In [ ]:
merged = feat_df[['txId', 'time_step']].merge(class_df, on='txId', how='left')
merged['class'] = merged['class'].fillna('unknown').astype(str)
merged['label'] = merged['class'].map({'1': 'Illicit', '2': 'Licit', 'unknown': 'Unknown'})

ts_counts = merged.groupby(['time_step', 'label']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(15, 5))
ts_counts.plot(kind='area', ax=ax, stacked=True,
               color={'Illicit': '#E84855', 'Licit': '#2DCC70', 'Unknown': '#95A5A6'},
               alpha=0.85)
ax.set_xlabel('Time Step', fontsize=12)
ax.set_ylabel('Transaction Count', fontsize=12)
ax.set_title('Transaction Distribution Across 49 Time Steps', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.set_xlim(1, 49)

plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_temporal_distribution.png', bbox_inches='tight')
plt.show()

## 4. Illicit Fraction per Time Step

In [ ]:
labelled = merged[merged['label'].isin(['Illicit', 'Licit'])].copy()
labelled['is_illicit'] = (labelled['label'] == 'Illicit').astype(int)

ts_fraud_rate = labelled.groupby('time_step')['is_illicit'].mean() * 100

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(ts_fraud_rate.index, ts_fraud_rate.values, color='#E84855', lw=2.5, marker='o', markersize=4)
ax.fill_between(ts_fraud_rate.index, ts_fraud_rate.values, alpha=0.2, color='#E84855')
ax.axhline(ts_fraud_rate.mean(), color='orange', linestyle='--', label=f'Mean: {ts_fraud_rate.mean():.1f}%')
ax.set_xlabel('Time Step'); ax.set_ylabel('Illicit Fraction (%)')
ax.set_title('Illicit Transaction Fraction per Time Step', fontweight='bold')
ax.legend(); ax.set_xlim(1, 49)
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_illicit_fraction.png', bbox_inches='tight')
plt.show()
print(f'Mean illicit fraction: {ts_fraud_rate.mean():.2f}%')
print(f'Peak illicit time-step: {ts_fraud_rate.idxmax()} ({ts_fraud_rate.max():.1f}%)')

## 5. Feature Statistics

In [ ]:
feat_cols = [c for c in feat_df.columns if c.startswith('feat_')]
stats = feat_df[feat_cols].describe().T
print(f'Feature dimension: {len(feat_cols)}')
print('\nSummary statistics (first 5 features):')
print(stats.head())

# Missing values
missing = feat_df[feat_cols].isnull().sum().sum()
print(f'\nTotal missing values: {missing} ({'none!' if missing == 0 else missing})')

## 6. Feature Correlation Heatmap (Local Features)

In [ ]:
# Use first 20 features for readability
local_feats = feat_cols[:20]
corr = feat_df[local_feats].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdYlBu_r', center=0, square=True,
            linewidths=0.3, cbar_kws={'shrink': 0.7}, ax=ax)
ax.set_title('Feature Correlation Matrix (Local Features 0–19)', fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_feature_correlation.png', bbox_inches='tight')
plt.show()

## 7. Graph Structure Analysis

In [ ]:
print(f'Total nodes: {feat_df.shape[0]:,}')
print(f'Total edges: {edge_df.shape[0]:,}')
print(f'Avg degree : {2 * edge_df.shape[0] / feat_df.shape[0]:.2f}')

# Degree distribution (sample)
from collections import Counter
all_nodes = list(edge_df['txId1']) + list(edge_df['txId2'])
degree_counts = Counter(all_nodes)
degrees = list(degree_counts.values())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(degrees, bins=50, color='#1A7AE8', edgecolor='white', log=True)
axes[0].set_xlabel('Degree'); axes[0].set_ylabel('Count (log scale)')
axes[0].set_title('Node Degree Distribution (log scale)', fontweight='bold')

axes[1].hist(degrees, bins=50, range=(0, 50), color='#E84855', edgecolor='white')
axes[1].set_xlabel('Degree'); axes[1].set_ylabel('Count')
axes[1].set_title('Node Degree Distribution (0–50, linear)', fontweight='bold')

plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_degree_distribution.png', bbox_inches='tight')
plt.show()

print(f'\nMax degree   : {max(degrees):,}')
print(f'Median degree: {np.median(degrees):.1f}')
print(f'Nodes with degree 1: {sum(1 for d in degrees if d == 1):,} ({100*sum(1 for d in degrees if d==1)/len(degrees):.1f}%)')

## 8. Feature Distributions — Illicit vs Licit

In [ ]:
merged_feats = feat_df.merge(class_df, on='txId', how='inner')
merged_feats = merged_feats[merged_feats['class'].isin(['1', '2'])].copy()
merged_feats['label'] = merged_feats['class'].map({'1': 'Illicit', '2': 'Licit'})

# Top 6 most discriminative features (by Jensen-Shannon divergence proxy)
top_feats = ['feat_0', 'feat_1', 'feat_2', 'feat_3', 'feat_4', 'feat_5']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(top_feats):
    ill = merged_feats[merged_feats['label'] == 'Illicit'][feat].dropna()
    lic = merged_feats[merged_feats['label'] == 'Licit'][feat].dropna()
    vmin = min(ill.quantile(0.05), lic.quantile(0.05))
    vmax = max(ill.quantile(0.95), lic.quantile(0.95))
    ill_clipped = ill.clip(vmin, vmax)
    lic_clipped = lic.clip(vmin, vmax)
    axes[i].hist(ill_clipped, bins=50, alpha=0.6, color='#E84855', label='Illicit', density=True)
    axes[i].hist(lic_clipped, bins=50, alpha=0.6, color='#2DCC70', label='Licit', density=True)
    axes[i].set_title(f'{feat}', fontweight='bold')
    axes[i].legend(fontsize=9)

plt.suptitle('Feature Distributions: Illicit vs Licit (clipped 5th–95th pct)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_feature_distributions.png', bbox_inches='tight')
plt.show()

## 9. Wavelet Temporal Encoding Visualization

In [ ]:
import pywt

T = 49
wavelet = pywt.Wavelet('db4')

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

for ax, ts in zip(axes, [5, 25, 45]):
    indicator = np.zeros(T)
    indicator[ts - 1] = 1.0
    coeffs = pywt.wavedec(indicator, wavelet, level=3, mode='periodization')
    all_coeffs = np.concatenate(coeffs)
    
    ax.stem(all_coeffs, markerfmt='C0o', linefmt='C0-', basefmt='k-', use_line_collection=True)
    ax.set_title(f'Wavelet Coefficients for Time-Step t={ts}', fontweight='bold')
    ax.set_xlabel('Coefficient Index'); ax.set_ylabel('Amplitude')
    
    # Annotate sub-bands
    boundaries = [0]
    for c in coeffs:
        boundaries.append(boundaries[-1] + len(c))
    labels = ['cA₃', 'cD₃', 'cD₂', 'cD₁']
    for start, end, label in zip(boundaries[:-1], boundaries[1:], labels):
        ax.axvspan(start, end - 0.5, alpha=0.1, color=f'C{labels.index(label)}')
        ax.text((start + end) / 2, ax.get_ylim()[1] * 0.85, label, ha='center', fontsize=9, color='gray')

plt.suptitle('Multi-Scale Wavelet (db4, L=3) Temporal Encoding', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'eda_wavelet_encoding.png', bbox_inches='tight')
plt.show()
print('✅ Wavelet basis captures different localities for different time-steps')

## 10. Summary Statistics for Research Paper

In [ ]:
n_total     = feat_df.shape[0]
n_illicit   = (class_df['class'] == '1').sum()
n_licit     = (class_df['class'] == '2').sum()
n_unknown   = (class_df['class'] == 'unknown').sum()
n_edges     = edge_df.shape[0]
n_timesteps = feat_df['time_step'].nunique()
avg_degree  = 2 * n_edges / n_total

print('=' * 50)
print('  ELLIPTIC BITCOIN DATASET STATISTICS')
print('=' * 50)
print(f'  Total transactions : {n_total:>12,}')
print(f'  Total edges        : {n_edges:>12,}')
print(f'  Time steps         : {n_timesteps:>12}')
print(f'  Illicit            : {n_illicit:>12,}  ({100*n_illicit/n_total:.1f}%)')
print(f'  Licit              : {n_licit:>12,}  ({100*n_licit/n_total:.1f}%)')
print(f'  Unknown            : {n_unknown:>12,}  ({100*n_unknown/n_total:.1f}%)')
print(f'  Node feature dim   : {feat_df.shape[1] - 2:>12}')
print(f'  Avg node degree    : {avg_degree:>12.2f}')
print(f'  Imbalance ratio    : {n_licit/n_illicit:>12.1f}:1 (licit:illicit)')
print('=' * 50)